# 01. 定义add算子（Ascend版本）

In [1]:
import tilelang
import tilelang.ascend.language as T


N = 2**20
dtype = "float32"

@tilelang.jit(out_idx=[-1])
def vector_add(N, mode="simt"):
    num_blocks, num_threads, tile, num_stages = 64, 2048, 8192, 2
    if N <= 0 or N % (tile * num_blocks):
        raise ValueError(f"N must be a positive multiple of {tile * num_blocks}")
    iterations = N // (tile * num_blocks)

    @T.prim_func
    def main(
        A: T.Tensor(shape=(N,), dtype=dtype), # type: ignore
        B: T.Tensor(shape=(N,), dtype=dtype), # type: ignore
        C: T.Tensor(shape=(N,), dtype=dtype), # type: ignore
    ):
        with T.Kernel(num_blocks) as bx:
            a = T.alloc_shared((tile,), "float32")
            b = T.alloc_shared((tile,), "float32")
            c = T.alloc_shared((tile,), "float32")
            T.annotate_buffer_versions({a: num_stages, b: num_stages, c: num_stages})
            for step in T.Pipelined(iterations, num_stages=num_stages):
                begin = (step * num_blocks + bx) * tile
                T.copy(A[begin : begin + tile], a)
                T.copy(B[begin : begin + tile], b)
                if mode == "simt":
                    with T.SimtVF(threads=num_threads):
                        for i in T.Parallel(tile):
                            c[i] = a[i] + b[i]
                elif mode == "simd_high":
                    with T.SimdVF():
                        for i in T.Parallel(tile):
                            c[i] = a[i] + b[i]
                else:
                    with T.SimdVF():
                        mask = T.simd.pset(32)
                        for i in range(tile // 64):
                            a_reg = T.simd.vld(a[i * 64])
                            b_reg = T.simd.vld(b[i * 64])
                            result = T.simd.vadd(a_reg, b_reg, mask)
                            T.simd.vsts(c[i * 64], result, mask)
                T.copy(c, C[begin : begin + tile])
    return main

my_jit_add = vector_add(N, mode="simd")
# my_jit_add = vector_add(N, mode="simd_high")
print(my_jit_add.get_kernel_source())

Loading tilelang libs from dev root: /data/caojian/workspace/git/github/tile-ai/tilelang/build
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/caojian/uv-envs/cjuv312_tile-ai_tilelang/lib/python3.12/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to prevent some unknown errors
  warnings.warn(


#include <tl_templates/ascend/common.h>
#include <tl_templates/ascend/debug.h>
#include <tl_templates/ascend/simd_inst.h>
__simd_vf__ inline void main_kernel_simd_vf_0(__ubuf__ uint8_t* buf_dyn_shmem, int32_t step) {
  vector_bool mask = asc_create_mask_b32(PAT_ALL);
  for (int32_t i = 0; i < 128; ++i) {
    auto a_reg = simd_inst::vlds_norm<float>((__ubuf__ float*)(&(((__ubuf__ float*)buf_dyn_shmem)[((step * 8192) + (i * 64))])), 0);
    auto b_reg = simd_inst::vlds_norm<float>((__ubuf__ float*)(&(((__ubuf__ float*)buf_dyn_shmem)[(((step * 8192) + (i * 64)) + 16384)])), 0);
    auto result = simd_inst::vadd(a_reg, b_reg, mask, MODE_ZEROING);
    simd_inst::vsts_norm(result, (__ubuf__ float*)(&(((__ubuf__ float*)buf_dyn_shmem)[(((step * 8192) + (i * 64)) + 32768)])), 0, mask);
  }
}

extern "C" __global__ __vector__ void main_kernel(__gm__ float* A, __gm__ float* B, __gm__ float* C) {
  asc_init();
  __ubuf__ uint8_t *buf_dyn_shmem = (__ubuf__ uint8_t *)0;
  asc_sync_notify(PIPE_MTE3, 

# 02. 运行&验证

In [2]:
import torch

torch.manual_seed(0)

a = torch.randn(N, ).npu()
b = torch.randn(N, ).npu()
c_golden = a + b

# print(f"Input tensor a: {a}")
# print(f"Input tensor b: {b}")
# print(f"Golden tensor c: {c_golden}")

c = my_jit_add(a, b)
# print(f"Output tensor c: {c}")

torch.testing.assert_close(c, c_golden, rtol=1e-5, atol=1e-5)
print("Test passed: The output tensor c is equal to the sum of the input tensors a and b.")


Test passed: The output tensor c is equal to the sum of the input tensors a and b.
